# Carlisle 2005: the River Eden flood against a full shallow-water reference

This notebook runs and analyses the real-river test of FloodA5: the January 2005
flood of the River Eden at Carlisle, where the Caldew and the Petteril join it.
It is the third of three notebooks, after `01_planar` (synthetic slopes) and
`02_channel` (a straight channel).

**The question.** How close is FloodA5's default **local-inertial** solver to the
**full shallow-water (SWE) reference** solver on a real river and floodplain? And
does the answer depend on the mesh? The event runs on two meshes of about the
same cell size:

- **A5 resolution 18**: pentagons of about 493 m², the area of a 22 m square;
- **Cartesian 20 m**: square cells aligned with the LiDAR, averaged to 20 m.

The **sub-grid (SGS)** solver also runs on A5. It was rewritten in 0.7.0
(`docs/HYDRAULICS.md` §9, `CHANGELOG.md`): §5.5 compares it with the other
22 m runs like for like, and §5.7 — optional, it needs two GPU runs — with
5 m references that resolve the river banks.

**Two cases from the case registry** (`cases/registry/carlisle.jl`;
`docs/TEST_CASES.md` §3):

| Case | Forcing | Compared at | Used for |
|---|---|---|---|
| `carlisle_peak` | the three river inflows only | the 6 h before the Eden's peak, 49.5–55.5 h | **the main comparison** (§5) |
| `carlisle` | the inflows **plus 10 mm/h of uniform rain**, an artificial test forcing | the first 6 h | the recorded test of HYDRAULICS §15.10–15.11 (§6) |

In both cases the river starts dry.

The 10 mm/h rain of `carlisle` is **not the 2005 event's rain**. It is a test
load from the solver-damping series (HYDRAULICS §15.6–15.11): 60 mm over the whole
domain in 6 h. Early in the event the rivers carry little water, so most of what
looks flooded at 6 h in that case is a rain-on-grid sheet over the floodplain,
not river flooding. §6 shows this directly.

**How to run it.** Use the kernel *Julia FloodA5 1.12* (`notebooks/README.md`)
and run all cells.

- **First run.** The notebook builds the meshes (about 7 min) and makes the runs
  in `test/case_runs/carlisle/` and `test/case_runs/carlisle_peak/`:
  - `carlisle`: about 50 min, with one 4-thread run at a time;
  - `carlisle_peak`: about 1.5 h for each of the two SWE runs over 55.5 h, at
    6 threads each, and about 15–25 min for each of the others.
- **Later runs.** A run is reused when its output records the same command line
  and a completed run, so the notebook then takes a few minutes.
- **Analysis only.** `RUN_CASES = false` analyses the existing runs without
  starting any.

## Setup

The notebook loads three modules, all from the repository:

| Module | File | Purpose |
|---|---|---|
| `FA` | `analysis/FloodA5Analysis.jl` | loads runs and computes metrics |
| `FC` | `cases/FloodA5Cases.jl` | the case registry and runner |
| `FP` | `visualisation/FloodA5Plots.jl` | static CairoMakie figures |

None of them loads the model or Python. The simulations run as separate
`julia FloodModel.jl …` processes. The repository root comes from the active
project (the kernel starts Julia with `--project=<repo>`), so the notebook works
from any checkout.

In [ ]:
const REPO = dirname(Base.active_project())
include(joinpath(REPO, "analysis", "FloodA5Analysis.jl"));   const FA = FloodA5Analysis
include(joinpath(REPO, "cases", "FloodA5Cases.jl"));         const FC = FloodA5Cases
include(joinpath(REPO, "visualisation", "FloodA5Plots.jl")); const FP = FloodA5Plots
using CairoMakie, Printf, Statistics, Logging
import ArchGDAL as AG
CairoMakie.activate!(type = "png")   # PNG: an SVG of 30 000 cell polygons would be huge
FP.activate!()                       # the FloodA5 figure style

RUN_CASES = true      # false: analyse existing runs only (an error if one is missing)
PARALLEL  = 1         # simulations at a time
THREADS   = nothing   # threads per simulation; nothing = each run's own (4; 6 for the long SWE runs)

c6 = FC.case("carlisle")          # first 6 h, inflows + 10 mm/h test rain
cpk = FC.case("carlisle_peak")     # 0–55.5 h, inflows only; compared over 49.5–55.5 h
CARL = joinpath(REPO, "test", "carlisle")        # the tracked inputs
T6 = 6 * 3600.0                                  # the end of `carlisle`
T_WIN = (49.5, 52.5, 55.5) .* 3600.0             # the `carlisle_peak` comparison times
T_PEAK = T_WIN[end]                              # the Eden's peak, 55.5 h
hrs(t) = @sprintf("%.1f h", t / 3600)

# Small HTML tables (an IJulia Markdown table also carries a LaTeX copy that VS Code cannot render)
function htmltable(header, rows; caption = "")
    h(x) = replace(string(x), "&" => "&amp;", "<" => "&lt;", ">" => "&gt;")
    io = IOBuffer()
    print(io, "<table>")
    isempty(caption) || print(io, "<caption style='text-align:left'>", h(caption), "</caption>")
    print(io, "<thead><tr>", join(("<th>" * h(x) * "</th>" for x in header)), "</tr></thead><tbody>")
    for r in rows
        print(io, "<tr>", join(("<td style='text-align:right'>" * h(x) * "</td>" for x in r)), "</tr>")
    end
    print(io, "</tbody></table>")
    return HTML(String(take!(io)))
end
f3(x) = isfinite(x) ? @sprintf("%.3f", x) : "–"
f2(x) = isfinite(x) ? @sprintf("%.2f", x) : "–"
println("repository: ", REPO)

## 1. The cases in the registry

In the registry, a case is plain data:

- the preparation steps (here, mesh generation);
- the runs, each one `FloodModel.jl` argument list;
- the metrics computed from the runs;
- the physical checks the metrics must pass;
- expected values for regression (`cases/expected/<case>.toml`).

`FC.show_case` prints the case's description and every command exactly as the
runner executes it, so the text and the commands cannot drift apart. Each command
is what you would type from the repository root, with paths expanded for this
checkout. `carlisle_peak` reuses the meshes that `carlisle` prepares.

In [ ]:
FC.show_case(cpk)

In [ ]:
FC.show_case(c6)

## 2. The inputs

The cases use four tracked files in `test/carlisle/`:

| File | Content |
|---|---|
| `Carlisle_domain.geojson` | the area of interest (AOI): a 4.75 km × 3.06 km rectangle in British National Grid, stored in WGS 84 |
| `Carlisle_LiDAR_5m_mean.tif` | 5 m LiDAR terrain, British National Grid with ODN heights |
| `carlisle_wgs84.bci` | the boundary conditions: where water enters (LISFLOOD-FP format) |
| `carlisle_wgs84.bdy` | the inflow hydrographs the `.bci` refers to |

FloodA5 works in longitude and latitude on the sphere. The DEM stays in its own
coordinate system: the model reprojects its sample points into it, not the raster.

### 2.1 Domain and DEM

In [ ]:
aoi = read(joinpath(CARL, "Carlisle_domain.geojson"), String)
println("AOI properties: ", match(r"(\x22MINX[^}]*?)\s*\}", aoi).captures[1])

AG.read(joinpath(CARL, "Carlisle_LiDAR_5m_mean.tif")) do ds
    gt = AG.getgeotransform(ds)
    band = AG.getband(ds, 1)
    z = Float64.(AG.read(band)); nd = AG.getnodatavalue(band)
    v = filter(x -> isfinite(x) && (nd === nothing || x != nd), z)
    @printf("DEM: %d × %d pixels of %.0f m; elevation %.1f to %.1f m (median %.1f m)\n",
            AG.width(ds), AG.height(ds), gt[2], minimum(v), maximum(v), median(v))
    println("CRS: ", match(r"^\w+\[\x22([^\x22]+)\x22", AG.getproj(ds)).captures[1])
end

### 2.2 River inflows

The `.bci` file has one line per boundary condition:

- **`P` lines** are point inflows: longitude, latitude, the code `QVAR` (a
  discharge that varies in time) and the name of the hydrograph in the `.bdy`
  file. The model adds each hydrograph to the cell nearest its point. The three
  points are the River Eden (`main`), the Caldew (`cummersdale`) and the
  Petteril (`harraby`).
- **The `W` line** is a LISFLOOD-FP "west edge" boundary. FloodA5 domains can be
  any polygon, so it ignores `N`/`E`/`S`/`W` lines with a warning (`--bc-file`
  places boundaries by location instead).

FloodA5 does not need the `W` line: every edge of the domain is already an open
boundary, a free outflow at normal depth (`--open-boundary normal`, the default
since 0.3.0; HYDRAULICS §15.11).

The `.bdy` file holds one block per hydrograph: the name, the number of rows,
the time unit, then rows of *value, time* (in that order: discharge first).
The cell below reads it with the model's own reader,
`boundaryinputs/timeseries_io.jl`.

In [ ]:
println(read(joinpath(CARL, "carlisle_wgs84.bci"), String))
println(join(first(readlines(joinpath(CARL, "carlisle_wgs84.bdy")), 7), "\n"), "\n…")

module BDY   # the model's .bdy reader, on its own
    include(joinpath(Main.REPO, "boundaryinputs", "timeseries_io.jl"))
end
series = with_logger(ConsoleLogger(stderr, Logging.Warn)) do
    BDY.read_timeseries(BDY.LisfloodBDYReader(joinpath(CARL, "carlisle_wgs84.bdy")))
end
RIVERS = ("main", "cummersdale", "harraby")
RIVER = Dict("main" => "Eden (main)", "cummersdale" => "Caldew (cummersdale)", "harraby" => "Petteril (harraby)")
qat(n, t) = (s = series[n]; k = findfirst(>=(t - 1), s[1]); s[2][k])
vol_to(n, T) = (t = series[n][1]; q = series[n][2]; sum((q[k] + q[k+1]) / 2 * (t[k+1] - t[k]) for k in 1:length(t)-1 if t[k+1] <= T + 1))
htmltable(["river", "Q(0)", "Q(6 h)", "Q(49.5 h)", "Q(55.5 h)", "peak", "peak at", "volume 0–6 h", "volume 0–55.5 h"],
          [[RIVER[n], f2(qat(n, 0)), f2(qat(n, T6)), f2(qat(n, T_WIN[1])), f2(qat(n, T_PEAK)),
            f2(maximum(series[n][2])), hrs(series[n][1][argmax(series[n][2])]),
            f3(vol_to(n, T6) / 1e6), f3(vol_to(n, T_PEAK) / 1e6)] for n in RIVERS];
          caption = "Inflow hydrographs (discharge in m³/s, volumes in Mm³)")

In [ ]:
let fig = Figure(size = (900, 420))
    ax = Axis(fig[1, 1]; xlabel = "Time since the start of the .bdy series (h)", ylabel = "Discharge (m³/s)",
              title = "River inflows, 0–60 h", subtitle = "shaded: the two comparison windows")
    vspan!(ax, [0.0, T_WIN[1] / 3600], [T6 / 3600, T_PEAK / 3600]; color = [(FP.SERIES[4], 0.25), (FP.SERIES[1], 0.18)])
    for (i, n) in enumerate(RIVERS)
        t, q = series[n]; s = t .<= 60 * 3600
        lines!(ax, t[s] ./ 3600, q[s]; color = FP.SERIES[i], linewidth = 2, label = RIVER[n])
    end
    text!(ax, 3.0, 1180; text = "carlisle\n0–6 h\n+ test rain", align = (:center, :top), fontsize = 11, color = FP.INK2)
    text!(ax, 52.5, 650; text = "carlisle_peak\nwindow\n49.5–55.5 h", align = (:center, :top), fontsize = 11, color = FP.INK2)
    xlims!(ax, 0, 60); ylims!(ax, 0, 1350)
    Legend(fig[2, 1], ax; orientation = :horizontal)
    fig
end

The two windows see very different rivers:

- **The first 6 h (`carlisle`).** The rising limb has barely started. The Eden
  goes from about 100 to 230 m³/s, while the Caldew and Petteril stay roughly
  steady.
- **The 6 h before the peak (`carlisle_peak`).** The Eden rises from about
  1 030 m³/s to its peak of 1 273 m³/s at 55.5 h. The Caldew and Petteril peaked
  earlier, at 38.5 and 44 h, and are receding.

The model has no start-time option, so `carlisle_peak` simulates the whole event
from t = 0, with the river initially dry, and is compared only in the window.

**Rain.** `carlisle_peak` has none. `carlisle` adds `--rainfall 10`, a uniform
10 mm/h on every cell for the whole 6 h. As explained above, this is a test load,
not the observed rain; §6 shows how much of that case's flooding it makes.

## 3. The meshes

`carlisle` prepares three meshes in `test/case_runs/carlisle/`; `carlisle_peak`
uses the same ones. A preparation step is skipped when its output files exist.

### 3.1 A5 resolution 18 (pya5)

`FloodModel.jl --meshgen AOI --meshres 18 --dem DEM --meshout MESH --mesh-only`
builds the mesh in three stages. It runs with `--threads 1` because of a
PyCall/GDAL hazard.

1. **Julia samples the AOI.** It lays a dense grid of points over the AOI's
   bounding box, at 0.45 of a cell width, and keeps the points inside the
   polygon. This point-in-polygon test runs on the GPU if one is available,
   otherwise on CPU threads.
2. **Python finds the cells** (`mesh/a5_bridge.py`, run with FloodA5's Python:
   PyCall's Conda environment, where pya5 is installed). In order:
   - pya5's `polygon_to_cells` (pya5 1.x) gives every resolution-18 cell that
     overlaps the AOI;
   - the set grows by each cell's neighbours whose polygon intersects the AOI,
     until nothing changes;
   - two cells are neighbours if they share two vertices, which is exact, unlike
     pya5's `grid_disk`;
   - each cell's area comes from its vertices.

   The bridge writes a GeoParquet file and records the cell-ID scheme
   (`a5-v1`) in its metadata.
3. **Julia samples the DEM.** Each cell's elevation is the mean of the DEM,
   interpolated bilinearly at up to 256 quasi-random (Halton) points inside the
   polygon (`A5Grid.sample_dem_mean!`). Cells with no DEM under them get NaN and
   take no part in the flow.

The core of stage 2:

In [ ]:
# Print a few lines of a source file, from the first line that matches `from`
function excerpt(file, from, n)
    L = readlines(joinpath(REPO, file))
    i = findfirst(l -> occursin(from, l), L)
    i === nothing && return println("(", file, ": ", repr(from), " not found)")
    println(file, ", lines ", i, "–", i + n - 1, ":\n")
    foreach(println, L[i:i+n-1])
end
excerpt("mesh/a5_bridge.py", "# ── Pass 1: seed cell set", 16)
println()
excerpt("mesh/a5_bridge.py", "# ── Pass 2: grid_disk expansion", 26)

### 3.2 SGS tables on the same A5 mesh

The SGS solver needs more than one elevation per cell. The SGS mesh is the same
A5 mesh, generated with `--flow-model sgs`, so that `A5Grid.build_sgs_tables!`
also stores two kinds of sub-grid table from the DEM.

- **Per cell:** the DEM is sampled on a regular working grid (about 1 600
  points per cell, finer than the 5 m pixel). It gives a 100-level storage
  table — stored volume and wetted area against water level, between the
  cell's lowest and highest ground — and the cell's *connectivity*: for each
  point, the level at which it joins the cell's main basin through ground
  below that level.
- **Per face:** the DEM is sampled every 2.5 m (half a pixel) along the face.
  One table per face, shared by its two cells, gives its flow area, wetted
  width and conveyance Σ dx·h^(5/3) at 64 levels from its sill up; a point of
  the face counts only once the water there is connected to the main basins
  of both cells.

The SGS solver measures depth from the cell's **lowest** ground and fills the
cell's low parts first, such as a channel narrower than the cell. Flow
between two cells follows the face's own ground profile: every strip of the
face carries its own Manning flow (HYDRAULICS §9.2).

### 3.3 Cartesian 20 m

`cases/scripts/average_dem.jl` averages the 5 m LiDAR onto a 20 m grid
(`gdalwarp -tr 20 20 -r average -tap`: each 20 m cell is the mean of its 16
pixels, on a grid snapped to multiples of 20 m). `--meshgen-cartesian` then
turns each pixel into a cell whose elevation is the pixel value. This step needs
no Python.

The A5 cells are about 23% larger in area than the Cartesian cells, so the two
meshes have similar, but not equal, resolution.

The next cell runs the missing preparation steps. It prints "up to date" for
steps whose outputs exist. Then it prints the key lines of each step's log.

In [ ]:
FC.prepare(c6)
println()
keylines = r"Sample points:|Points inside AOI|Python bridge:|Mesh generated in|outside DEM extent|mean sampling complete|SGS pre-processing:|edge sample spacing|SGS tables built in \d|Raster-aligned Cartesian grid:|cells of \d+ m"
for p in c6.prep
    log = joinpath(FC.case_dir(c6), "prep_$(p.name).log")
    isfile(log) || continue
    println("── ", p.name)
    for l in eachline(log)
        occursin(keylines, l) || continue
        l = replace(l, r"[A-Za-z]:[\\/][^\s:]*[\\/]" => "", r"^[┌└│\[ ]*(Info:|Warning:)\s*" => "")
        println("   ", strip(l))
    end
end

The figure shows the two meshes' elevations, read from their GeoParquet files.
The zoom covers the Caldew joining the Eden:

- the A5 cells follow no axis, and a channel crosses them at every angle;
- the Cartesian grid staircases every channel that is not aligned with x or y.

The narrow Caldew and Petteril are only one or two cells wide on both meshes.

In [ ]:
a5_mesh   = FA.load_mesh_parquet(joinpath(FC.case_dir(c6), "carlisle_a5_res18.parquet"))
cart_mesh = FA.load_mesh_parquet(joinpath(FC.case_dir(c6), "carlisle_cart20.parquet"))
for (name, m) in (("A5 res 18", a5_mesh), ("Cartesian 20 m", cart_mesh))
    @printf("%-15s %6d cells, mean area %5.1f m² (a %.1f m square), domain %.2f km², %d cells without elevation\n",
            name, length(m), mean(m.area), sqrt(mean(m.area)), sum(m.area) / 1e6, count(isnan, m.elevation))
end

# A map of a mesh alone: FP draws runs, so wrap the mesh in a one-frame Run
mesh_run(m, label) = FA.Run("", label, 2, [0.0], m, Dict{String,Any}(), "")
let fig = Figure(), rng = (8.0, 32.0), box = (-2.9480, -2.9380, 54.8985, 54.9045)
    for (j, r) in enumerate((mesh_run(a5_mesh, "A5 res 18"), mesh_run(cart_mesh, "Cartesian 20 m")))
        FP.fieldmap!(fig[1, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                     basemap = :none, title = r.label, subtitle = "cell elevation, $(length(r.mesh)) cells",
                     width = 430, colorbar = j == 2)
        p = FP.fieldmap!(fig[2, j], r, 1; var = :elevation, colormap = :batlowW, colorrange = rng, sources = false,
                         basemap = :none, title = r.label * ": zoom", subtitle = "the Caldew joining the Eden",
                         width = 430, colorbar = false)
        limits!(p.axis, box...)
        p.axis.aspect = AxisAspect((box[2] - box[1]) * cosd(54.9) / (box[4] - box[3]))
    end
    resize_to_layout!(fig)
    fig
end

## 4. The runs and the checks

Both cases make the same five runs; `carlisle` adds two runs without rain.

| Run | Mesh | Solver | Role |
|---|---|---|---|
| `a5_swe` | A5 | `--flow-model swe` | the reference on A5 (see below) |
| `a5_li` | A5 | `--flow-model standard` | the 0.3.0 local-inertial default; compared with `a5_swe` |
| `cart_swe` | Cartesian | `--flow-model swe` | the reference on Cartesian |
| `cart_li` | Cartesian | `--flow-model standard` | local-inertial; compared with `cart_swe` |
| `a5_sgs` | A5 + SGS tables | `--flow-model sgs` | the sub-grid solver |
| `a5_li_norain`, `cart_li_norain` | A5, Cartesian | `--flow-model standard` | `carlisle` only: the same 6 h without the test rain |

**The SWE reference** solves the full 2-D shallow-water equations, advection
included. It is a second-order HLLC finite-volume scheme with hydrostatic
reconstruction (HYDRAULICS §15.10). It runs on the same mesh, bed, sources and
boundaries as the local-inertial solver, so the comparison isolates what the
local-inertial simplification does on that mesh. In the 6 h test it converges in
its own numerical parameters: halving its Courant number or adding eddy viscosity
changes its storage by less than 0.1%.

**The SGS run** has no reference with a sub-grid bed on this mesh. Against
the 22 m SWE run it is compared like for like — stored volume, water level
where both are wet, wetted area — and the case checks it against the bar
agreed for it (§5.5); §5.7 compares it with 5 m references.

**Options:**

| Option | `carlisle_peak` | `carlisle` |
|---|---|---|
| `--inflow-bci` | the three hydrographs | the same |
| `--rainfall` | none | `10` (10 mm/h on every cell; test forcing) |
| `--sim-duration` | `199800` (55.5 h) | `21600` (6 h) |
| `--dt-max` | `10` (adaptive step, at most 10 s) | `10` |
| `--output-interval` | `1800` (a frame every 30 min) | `900` (every 15 min) |

The runner appends `--output test/case_runs/<case>/runs/<run>.h5` and writes a
log next to it. Every output file records its command
(`FO.run_info(path)["argv"]`), and `julia --project=. FloodModel.jl --rerun <run>.h5`
reproduces a run bit for bit.

The next cells print the commands, then run both cases: they make the runs that
are missing or out of date, analyse them and check them. With `RUN_CASES = false`
they only analyse and check the existing runs.

Each case's checks are listed at the end of its `FC.show_case` output above.
Every metric is also compared with `cases/expected/<case>.toml`, a regression
test. A5 results are sensitive to round-off in the bed. A mesh regenerated on
another machine can move them beyond those tight tolerances while the physical
checks still pass; if so, re-record the expected values
(`cases/run.jl baseline <case>`).

In [ ]:
for c in (cpk, c6), j in c.jobs
    println(c.name, " / ", j.name, "   (", join(("$k = $v" for (k, v) in j.tags), ", "), ")")
    println("  julia --project=. --threads ", j.threads, " FloodModel.jl ", join(FC.job_argv(c, j), " "), "\n")
end

In [ ]:
t0 = time()
ok6 = RUN_CASES ? FC.run_case(c6; parallel = PARALLEL, threads = THREADS) : FC.check(c6)   # also prepares the meshes
println()
okp = RUN_CASES ? FC.run_case(cpk; parallel = PARALLEL, threads = THREADS) : FC.check(cpk)
@printf("\n%s (%.1f min)\n", ok6 && okp ? "All checks pass." : "SOME CHECKS FAIL: see above.", (time() - t0) / 60)

In [ ]:
m6 = FC.analyse(c6; io = devnull)
mp = FC.analyse(cpk; io = devnull)
load_case(c, label) = Dict(j.name => FA.load_run(FC.job_output(c, j); label = label[j.name]) for j in c.jobs)
LABEL = Dict("a5_swe" => "A5 SWE", "a5_li" => "A5 local-inertial", "cart_swe" => "Cartesian SWE",
             "cart_li" => "Cartesian local-inertial", "a5_sgs" => "A5 SGS",
             "a5_li_norain" => "A5 local-inertial, no rain", "cart_li_norain" => "Cartesian local-inertial, no rain")
P = load_case(cpk, Dict(k => v * " (river only)" for (k, v) in LABEL))
S = load_case(c6, Dict(k => v * (endswith(k, "norain") ? "" : " (+ test rain)") for (k, v) in LABEL))
ORDER = ["a5_swe", "a5_li", "cart_swe", "cart_li", "a5_sgs"]
# The SGS storage tables (wetted area against level) are in the mesh file, not the output:
# FA.sgs_tables reads them in the run's cell order and checks they belong to it
SGS_MESH = joinpath(FC.case_dir(c6), "carlisle_a5_res18_sgs.parquet")
SGS_TABLES = FA.sgs_tables(P["a5_sgs"]; parquet = SGS_MESH)
"Wetted area (m²) of run `r` at frame `k` with local depth > `wet`: SGS from its tables, cell-mean runs whole cells."
wet_area(r, k; wet = 0.1) = sum(r.mesh.area .* FA.wetted_fraction(r, k; wet,
                                tables = r.flow_model == "sgs" ? SGS_TABLES : nothing))

# run cost and volumes, from each output file's end-of-run record
cost(R, n) = (s = R[n].info["summary"]; [n, f3(s["vol_added_m3"] / 1e6), f3(s["vol_removed_m3"] / 1e6),
              @sprintf("%.1e", abs(s["conservation_error_m3"]) / s["vol_applied_m3"]), string(s["n_steps"]),
              @sprintf("%.1f", s["wall_simulation_s"] / 60)])
hdr = ["run", "added (Mm³)", "removed (Mm³)", "conservation error (rel.)", "steps", "wall (min)"]
display(htmltable(hdr, [cost(P, n) for n in ORDER]; caption = "carlisle_peak (0–55.5 h, river only): volumes at 55.5 h and cost"))
display(htmltable(hdr, [cost(S, n) for n in vcat(ORDER, ["a5_li_norain", "cart_li_norain"])];
                  caption = "carlisle (0–6 h; all but *_norain with the 10 mm/h test rain): volumes at 6 h and cost"))

How to read these tables:

- **Mass.** The conservation error, |added − stored − removed| ÷ added, is at
  round-off level in every run.
- **Cost.** The SWE runs take many times the steps of the local-inertial runs on
  the same mesh, because their stable step is bounded by the wave speed
  |u| + √(gh) rather than by √(gh) alone. The `wall` column is the simulation
  time of each run as it was made, on a shared machine.

## 5. The 6 h before the Eden's peak (`carlisle_peak`: river inflow only, 49.5–55.5 h)

### 5.1 Storage and flooded area

The plot shows the water stored in the domain at each 30-minute frame, over the
whole run; the comparison window is shaded. Colour marks the mesh (A5 blue,
Cartesian orange) and line style the solver (local-inertial solid, SWE dashed,
SGS dotted). The SGS storage is the sum of the cell volumes (`FA.storage`).

In [ ]:
PRUNS = [P[n] for n in ("a5_li", "a5_swe", "a5_sgs", "cart_li", "cart_swe")]
let fig = Figure(size = (820, 480))
    ax = FP.storage_plot!(fig[1, 1], PRUNS; title = "Water stored, carlisle_peak (river inflow only, no rain)")
    vspan!(ax, [T_WIN[1] / 3600], [T_PEAK / 3600]; color = (FP.MUTED, 0.18))
    text!(ax, (T_WIN[1] + T_PEAK) / 7200, 0; text = "window", align = (:center, :bottom), offset = (0, 6),
          fontsize = 11, color = FP.INK2)
    fig
end

In [ ]:
# flooded area: the case's metric (whole cells deeper than 0.1 m) for the cell-mean runs; for SGS the
# area its tables put more than 0.1 m under water (the case's SGS metric counts whole cells by SGS depth,
# which is measured from each cell's lowest ground: see §5.5)
peak_area(n, t) = n == "a5_sgs" ? wet_area(P[n], FA.frame_at(P[n], t; tol = 300)) / 1e6 :
                                  mp[n]["wet_area_0p1_$(Dict(T_WIN[1] => "w0", T_WIN[2] => "wmid", T_WIN[3] => "w1")[t])_km2"]
rows = [vcat([P[n].label], [f3(mp[n]["storage_$(t)_m3"] / 1e6) for t in ("w0", "wmid", "w1")],
             [f2(peak_area(n, t)) for t in T_WIN]) for n in ORDER]
display(htmltable(["run", "storage 49.5 h", "52.5 h", "55.5 h", "area > 0.1 m, 49.5 h", "52.5 h", "55.5 h"], rows;
                  caption = "carlisle_peak: storage (Mm³) and flooded area deeper than 0.1 m (km²) in the window (SGS: from its tables)"))
rows = [vcat([P[n].label], [f3(mp[n]["storage_ratio_$t"]) for t in ("w0", "wmid", "w1")],
             [f3(mp[n]["rms_dh_$(t)"]) for t in ("w0", "wmid", "w1")],
             [f2(mp[n]["csi_$t"]) for t in ("w0", "wmid", "w1")], [f2(mp[n]["bias_$t"]) for t in ("w0", "wmid", "w1")])
        for n in ("a5_li", "cart_li")]
display(htmltable(["run", "storage ÷ SWE 49.5 h", "52.5 h", "55.5 h", "rms Δh (m) 49.5 h", "52.5 h", "55.5 h",
                   "CSI 49.5 h", "52.5 h", "55.5 h", "bias 49.5 h", "52.5 h", "55.5 h"], rows;
                  caption = "carlisle_peak: local-inertial against SWE on the same mesh (FA.compare: rms over cells wetter than 1 cm in either; CSI and bias at 0.1 m)"))
# Mean speed where the reference is at least 3 m deep (FA.compare's speed_ratio): not a case metric
rows = Vector{String}[]
for (ref, run) in (("a5_swe", "a5_li"), ("cart_swe", "cart_li")), t in T_WIN
    c = FA.compare(P[ref], P[run]; k_ref = FA.frame_at(P[ref], t; tol = 300))
    push!(rows, [P[run].label, hrs(t), f2(c.speed_ref), f2(c.speed), f2(c.speed_ratio), string(c.n_deep)])
end
htmltable(["run", "time", "SWE speed (m/s)", "local-inertial speed (m/s)", "ratio", "cells ≥ 3 m deep in SWE"], rows;
          caption = "carlisle_peak: mean speed over the cells at least 3 m deep in the reference")

**Near the peak the result is the opposite of the 6 h test.** In the window both
local-inertial runs store *less* than the reference, not more.

| | A5 | Cartesian |
|---|---|---|
| local-inertial storage ÷ SWE | 0.74–0.76 | about 0.85 |
| storage at 55.5 h, local-inertial vs SWE (Mm³) | 8.01 vs 10.51 | 7.92 vs 9.36 |
| CSI | 0.86–0.87 | 0.92–0.93 |
| bias | 0.87–0.89 | 0.93–0.94 |

- **The local-inertial runs let more water out.** By 55.5 h the A5
  local-inertial run has let out 149.6 Mm³, against 147.1 Mm³ for the A5
  reference. That 2.5 Mm³ is the storage gap.
- **They flood less.** Their flooded area is 4.57 km² on both meshes, against
  5.16 (A5 SWE) and 4.91 km² (Cartesian SWE). A bias below 1 means the
  local-inertial run has fewer wet cells than the reference.
- **Their deep water moves faster** than the reference's: 1.3–1.4 times as fast
  on A5, and 1.15 times on Cartesian.
- **The mesh matters for the reference, not for local-inertial.** The two
  local-inertial runs are nearly identical (8.01 and 7.92 Mm³, 4.57 km² each),
  while the two references differ by 12% (10.51 and 9.36 Mm³). The A5 reference's
  deep water is also the slowest of the four runs (about 1.1 m/s, against 1.5 m/s
  for the Cartesian reference).
- **The gap is not only at the peak.** The storage plot shows the local-inertial
  runs below their references from early in the event on A5, and from about 10 h
  on Cartesian.

### 5.2 Depth maps in the window: local-inertial against SWE

Depth at 49.5, 52.5 and 55.5 h (river inflow only). Cells shallower than 0.1 m
show the terrain. The stars mark the three inflow points.

In [ ]:
DMAX = 7.0   # one colour range for all the peak-window maps (m)
FP.fieldmaps([P["a5_li"], P["a5_swe"]], collect(T_WIN); dry_threshold = 0.1, colorrange = (0, DMAX), panel = 330)

In [ ]:
FP.fieldmaps([P["cart_li"], P["cart_swe"]], collect(T_WIN); dry_threshold = 0.1, colorrange = (0, DMAX), panel = 330)

At this scale the four floods look alike. In every run the Eden fills its
floodplain between the meanders, and the Caldew and Petteril valleys flood. The
differences are in depth (§5.3) and at the edges of the flood. One example is the
floodplain inside the Eden's north-western loop: the A5 reference floods it from
about 32 h and the Cartesian reference from about 46 h, but the local-inertial
runs only inside the window (§5.4).

### 5.3 Differences at the peak (55.5 h)

The two upper maps show local-inertial minus SWE depth at 55.5 h, on cells wetter
than 1 cm in either run, so their rms is the rms Δh of the table. Blue means the
local-inertial run is deeper. The extent maps classify each cell at the 0.1 m
threshold as wet in both (a hit), wet in the reference only (a miss), or wet in
the local-inertial run only (a false alarm).

In [ ]:
let fig = Figure()
    FP.difference_map!(fig[1, 1], P["a5_swe"], P["a5_li"]; t = T_PEAK, colorrange = (-1.5, 1.5), width = 460,
                       title = "A5: local-inertial − SWE, 55.5 h, river only")
    FP.difference_map!(fig[1, 2], P["cart_swe"], P["cart_li"]; t = T_PEAK, colorrange = (-1.5, 1.5), width = 460,
                       title = "Cartesian: local-inertial − SWE, 55.5 h, river only")
    resize_to_layout!(fig)
    fig
end

In [ ]:
let fig = Figure()
    FP.extent_map!(fig[1, 1], P["a5_swe"], P["a5_li"]; t = T_PEAK, width = 430)
    FP.extent_map!(fig[2, 1], P["cart_swe"], P["cart_li"]; t = T_PEAK, width = 430)
    resize_to_layout!(fig)
    fig
end

Where is the extra (or missing) water? The next cell splits the storage
difference Σ (h − h_SWE)·A by the reference's depth in each cell.

In [ ]:
function storage_by_class(ref, run, t)
    kr, kk = FA.frame_at(ref, t), FA.frame_at(run, t)
    href, h, A = FA.depth(ref, kr), FA.depth(run, kk), ref.mesh.area
    rows = Vector{String}[]
    for (name, lo, hi) in (("dry or < 0.1 m in SWE", -Inf, 0.1), ("0.1–1 m", 0.1, 1.0),
                           ("1–3 m", 1.0, 3.0), ("> 3 m (river channels)", 3.0, Inf), ("all cells", -Inf, Inf))
        sel = [i for i in eachindex(h) if isfinite(h[i]) && isfinite(href[i]) && lo < href[i] <= hi]
        push!(rows, [name, string(length(sel)), @sprintf("%+.3f", sum((h[i] - href[i]) * A[i] for i in sel; init = 0.0) / 1e6),
                     isempty(sel) ? "–" : @sprintf("%+.3f", mean(h[sel] .- href[sel]))])
    end
    return htmltable(["SWE depth class", "cells", "Δ storage (Mm³)", "mean Δh (m)"], rows;
                     caption = run.label * " − " * ref.label * " at " * FP.hhmm(run.t[kk]))
end
display(storage_by_class(P["a5_swe"], P["a5_li"], T_PEAK))
display(storage_by_class(P["cart_swe"], P["cart_li"], T_PEAK))

**The deficit is in water that the reference already holds.** Almost none of it
is on cells that are dry in the reference. By SWE depth class:

| SWE depth class | A5 | Cartesian |
|---|---|---|
| floodplain 1–3 m deep (the largest share) | −1.64 Mm³, 0.54 m shallower on average | −0.96 Mm³, 0.34 m shallower |
| channels deeper than 3 m | −0.45 Mm³, 0.59 m shallower | −0.19 Mm³, 0.30 m shallower |
| shallow edges (0.1–1 m) | −0.41 Mm³ | −0.29 Mm³ |

**The local-inertial water surface is lower over the whole flood:** by about
half a metre on A5 and a third of a metre on Cartesian. The difference maps show
this as a near-uniform red over the floodplain, darkest in A5's north-western
loop. The local-inertial run is deeper only in small patches, such as the
Cartesian Eden at the western outflow edge and parts of the Caldew and Petteril
valleys.

### 5.4 Time series at a river cell and a floodplain point

Two points, marked on the 55.5 h map:

- **River Eden:** a point in the channel at the southern end of its central meanders.
- **Floodplain:** a point inside the Eden's north-western meander loop, which
  floods during the window.

The time series cover the whole `carlisle_peak` run (river inflow only), with the
comparison window shaded. Each run uses its own cell nearest each point. The SGS run is left out, because
its depth is measured from each cell's lowest point and so is not comparable with
the others.

In [ ]:
POINTS = [(lon = -2.93647, lat = 54.90020, label = "River Eden channel"),
          (lon = -2.95000, lat = 54.90400, label = "Floodplain in the north-western loop")]
foreach(p -> @printf("%-34s lon %.5f  lat %.5f\n", p.label, p.lon, p.lat), POINTS)
let fig = Figure()
    p = FP.fieldmap!(fig[1, 1], P["a5_swe"], nothing; t = T_PEAK, dry_threshold = 0.1, colorrange = (0, DMAX), width = 560)
    FP.mark_points!(p, POINTS; labels = [q.label for q in POINTS])
    resize_to_layout!(fig)
    fig
end

In [ ]:
let fig = FP.timeseries([P[n] for n in ("a5_li", "a5_swe", "cart_li", "cart_swe")], POINTS;
                        labels = ["A5 local-inertial", "A5 SWE", "Cartesian local-inertial", "Cartesian SWE"])
    for ax in filter(x -> x isa Axis, fig.content)
        vspan!(ax, [T_WIN[1] / 3600], [T_PEAK / 3600]; color = (FP.MUTED, 0.18))
    end
    display(fig)
end
rows = [vcat([P[n].label], [f2(FA.depth(P[n], FA.frame_at(P[n], t))[FA.source_cell(P[n].mesh, p.lon, p.lat)])
                            for p in POINTS for t in (T_WIN[1], T_PEAK)]) for n in ("a5_li", "a5_swe", "cart_li", "cart_swe")]
htmltable(["run", "Eden 49.5 h", "Eden 55.5 h", "floodplain 49.5 h", "floodplain 55.5 h"], rows;
          caption = "carlisle_peak: depth (m) at the two points")

**In the Eden channel** all four runs fill in the first 2–3 h and then follow the
hydrograph:

- The A5 reference stands highest throughout: about 1 m above the others for
  most of the event, and 0.3–0.6 m at the peak (7.67 m against 7.37 m for the
  Cartesian reference and about 7.1 m for both local-inertial runs).
- The two local-inertial runs stay within a few centimetres to a few decimetres
  of each other.

**On the floodplain inside the north-western loop** the bank is overtopped at
different times:

| Run | Bank overtopped at | Depth at 55.5 h |
|---|---|---|
| A5 reference | about 32 h | 1.84 m |
| Cartesian reference | about 46 h | 1.15 m |
| both local-inertial runs | about 50 h, inside the window | 0.71–0.85 m |

The A5 reference's higher river stage reaches the floodplain first.

### 5.5 The SGS run at the peak

**SGS and cell-mean models do not mean the same by "depth".** An SGS cell's
depth is measured from its *lowest* ground, so an SGS cell with a little
water in a sub-grid channel has a depth of a metre or more and counts as
"wet", where a cell-mean model holding the same volume spreads it as a film
over the whole cell, or is dry there. Depth maps, depth differences and
counts of wet cells of SGS and cell-mean runs therefore compare different
things, and so does `FP.difference_map(…; var = :wse)` between them (it masks
each run by its own depth). The comparisons below use quantities that mean
the same in every model:

- **stored volume**, and the mean depth volume ÷ cell area;
- **wetted area** at 0.1 m local depth: SGS — the area of each cell its
  storage table puts more than 0.1 m under water (`FA.wetted_fraction`);
  cell-mean models — the whole area of cells deeper than 0.1 m;
- **water level**, only on cells wet in both runs (SGS: at least half the
  cell wetter than 0.1 m), where the two levels are levels of the same water.

`FA.compare_to_fine` computes them (it aggregates a fine run onto a coarse
mesh, §5.7; given a run on the same mesh, each cell is its own aggregate).

**What changed in 0.7.0.** Until 0.6.0 SGS stored 2.5 times the SWE
reference here, the Eden standing metres above the other solvers: its face
flow areas stopped growing above each cell's highest DEM sample, and its
donor limiter held channels brim-full (notebook 4, HYDRAULICS §15.13). It now
has one conveyance table per face continued above the profile, positivity
limiting, a time step that follows the faces, and connectivity (§3.2). The
0.6.0 values are kept in `cases/expected/v0.6.0/`.

In [ ]:
p060 = FC.TOML.parsefile(joinpath(REPO, "cases", "expected", "v0.6.0", "carlisle_peak_metrics.toml"))
WTAGS = ("w0", "wmid", "w1")
rows = vcat([vcat(["A5 SGS, 0.6.0 scheme"], [f3(p060["a5_sgs"]["storage_$(w)_m3"] / 1e6) for w in WTAGS])],
            [vcat([P[n].label], [f3(mp[n]["storage_$(w)_m3"] / 1e6) for w in WTAGS]) for n in ("a5_sgs", "a5_li", "a5_swe")])
display(htmltable(["run", "storage 49.5 h", "52.5 h", "55.5 h"], rows; caption = "carlisle_peak on A5: storage (Mm³), SGS before and after 0.7.0"))

# like for like against the 22 m SWE run on the same cells
own = collect(1:length(P["a5_swe"].mesh))
lfl = Dict((n, t) => FA.compare_to_fine(P["a5_swe"], P[n], t; owner = own, tables = n == "a5_sgs" ? SGS_TABLES : nothing)
           for n in ("a5_li", "a5_sgs"), t in T_WIN)
rows = [[P[n].label, hrs(t), f3(c.storage_ratio), @sprintf("%+.2f", c.level_median), f2(c.level_rms), string(c.n_level),
         f2(c.wet_area / 1e6), f2(c.wet_area_fine / 1e6), f2(c.csi)]
        for n in ("a5_li", "a5_sgs") for t in T_WIN for c in (lfl[(n, t)],)]
display(htmltable(["run", "time", "storage ÷ SWE", "level − SWE, median (m)", "rms (m)", "cells wet in both",
                   "wetted area (km²)", "SWE (km²)", "fuzzy CSI"], rows;
                  caption = "carlisle_peak, A5: local-inertial and SGS against the 22 m SWE run, like for like (FA.compare_to_fine; wet: 0.1 m local depth)"))

# the case's own SGS metrics (_carl_sgs_compare): wet = mean depth V/A ≥ 0.1 m; level rms over cells wet in both
rows = [[P[n].label, hrs(t), f3(mp[n]["storage_ratio_$w"]), haskey(mp[n], "mean_dh_$w") ? @sprintf("%+.2f", mp[n]["mean_dh_$w"]) : "–",
         f2(mp[n]["rms_dh_$w"]), f2(mp[n]["csi_$w"]), f2(mp[n]["bias_$w"])]
        for n in ("a5_li", "a5_sgs") for (w, t) in zip(WTAGS, T_WIN)]
display(htmltable(["run", "time", "storage ÷ SWE", "mean Δ level (m)", "rms Δ (m)", "CSI", "bias"], rows;
                  caption = "carlisle_peak, A5: the case's metrics against SWE (local-inertial: FA.compare, depth on one bed; SGS: water level and mean depth V/A ≥ 0.1 m)"))
for n in ("a5_swe", "a5_li", "a5_sgs")
    @printf("%-28s let out by 55.5 h: %.1f Mm³\n", P[n].label, P[n].info["summary"]["vol_removed_m3"] / 1e6)
end

The map shows the water level of each run minus the SWE run's at 55.5 h, on
the cells wet in both (as defined above), with one colour range: red where
the run stands lower than the reference.

In [ ]:
"Level of `run` minus level of `ref` at time t on the cells wet in both (wetted fraction ≥ ½ at 0.1 m), NaN elsewhere."
function level_diff(ref, run, t)
    kr, kk = FA.frame_at(ref, t; tol = 300), FA.frame_at(run, t; tol = 300)
    fr = FA.wetted_fraction(ref, kr); fk = FA.wetted_fraction(run, kk; tables = run.flow_model == "sgs" ? SGS_TABLES : nothing)
    ηr, ηk = FA.wse(ref, kr), FA.wse(run, kk)
    return [fr[i] >= 0.5 && fk[i] >= 0.5 && isfinite(ηr[i]) && isfinite(ηk[i]) ? ηk[i] - ηr[i] : NaN for i in eachindex(ηr)]
end
let fig = Figure()
    for (j, n) in enumerate(("a5_li", "a5_sgs"))
        d = level_diff(P["a5_swe"], P[n], T_PEAK); g = P[n].mesh
        dm = FA.MeshGeom(g.ids, g.lon, g.lat, d, g.area, g.polygons, g.neighbours, g.static, g.id_scheme)
        w = filter(isfinite, d)
        FP.fieldmap!(fig[1, j], mesh_run(dm, ""), 1; var = :elevation, colormap = FP.DIVERGING, colorrange = (-2, 2),
                     basemap = :grey, sources = false, width = 460, colorbar = j == 2, colorbar_label = "Δ water level (m)",
                     title = (n == "a5_li" ? "A5 local-inertial" : "A5 SGS") * " − A5 SWE, 55.5 h",
                     subtitle = @sprintf("cells wet in both · median %+.2f m, rms %.2f m", median(w), sqrt(mean(abs2, w))))
    end
    resize_to_layout!(fig)
    fig
end

**What the 22 m comparison shows.**

- **SGS is now in the same range as the other solvers**, where until 0.6.0
  it stood metres higher: it stores a little over half of the 22 m SWE
  run's volume through the window (about 0.52–0.57; local-inertial
  0.74–0.76), against 2.5 times before. Its water surface lies about 0.9 m
  below SWE's on the cells wet in both (local-inertial about 0.5 m), over
  most of the floodplain, and it wets less area (about 3.3 km² at the peak,
  against 4.6 for local-inertial and 5.2 for SWE). It lets more water out
  than either.
- **It is smooth and converged in its time step.** At the Courant number of
  the other solvers (0.7) its water surface broke into cell-scale
  oscillations in the deep, fast flow near the peak; at 0.5 and 0.35 it is
  smooth and the two agree (5.94 and 5.96 Mm³), so 0.5 is the SGS default
  (plan §9.3, `CHANGELOG.md`; those trial runs are not part of the case).
- **SGS conveys more than local-inertial, which already conveys more than
  SWE.** The case's check of the bar agreed for SGS — *no further from the
  SWE reference than local-inertial on the same mesh* — therefore fails; it
  is marked an **expected failure** (XFAIL in the checks of §4).
- **What this comparison cannot say.** The reference is SWE on cell-mean
  beds at the same 22 m. SGS differs from local-inertial only in the bed:
  each face conveys Σ h^(5/3) over its true profile, which is never less
  than the conveyance of the mean depth (a face half 6 m deep and half dry
  conveys 60% more than the same face 3 m deep throughout). A model that
  resolves the banks would side with SGS on that; SWE's momentum losses pull
  the other way. Neither 22 m run can decide which is nearer the truth: §5.7
  compares all three with 5 m runs.

### 5.6 What the peak window says

- **The early-window result does not carry over.** In the first 6 h with the test
  rain, local-inertial stored 1.25× SWE on A5 and 0.99× on Cartesian. Near the
  peak, with the river alone, it stores 0.76× and 0.85×. Both the sign and the size
  of its error depend on the flow.
- **Near the peak the river is deep and fast, and the local-inertial runs convey
  more.** Their deep water moves faster, they let more water out, and their water
  stands lower everywhere.
- **A plausible mechanism.** The local-inertial equations leave out the
  advection of momentum. The full shallow-water equations keep it, and with
  it the momentum losses where the flow bends, expands and spills onto the
  floodplain. Around the Eden's tight meanders at 1 000–1 300 m³/s, those
  losses could be what holds the reference's water higher. On the 22 m
  meshes this cannot be separated from the solvers' different bed
  representations; the 5 m runs of §5.7, if made, can separate them.
- **In the early window the flow is different.** It is a small river plus thin
  rain-on-grid films, where the step bed's water held in shallow hollows dominates
  (§6). So the error had the opposite sign there.
- **The mesh.** Near the peak the local-inertial result hardly depends on the mesh,
  but the reference does: the A5 reference stores 12% more than the Cartesian
  one, its deep water is slower, and its Eden stands higher throughout.
  - The straight-channel series (HYDRAULICS §15.12) found that the SWE solver on
    A5 conveys worst in channels running 0–15° from east at that series' test
    site.
  - The Eden here runs broadly east–west. Whether that is the cause is untested.
- **SGS** conveys more than both at 22 m (§5.5): about 0.55 of the SWE
  run's storage, its water ~0.9 m lower.
- **What to take from it.**
  - At the peak, the local-inertial default is within 15% of the reference's
    storage on Cartesian (CSI 0.92) and within 25% on A5 (CSI 0.87).
  - A single comparison window does not characterise the solver, least of all
    one dominated by a test rain.
  - A 22 m SWE run is a reference for what the local-inertial
    simplification does on that mesh, not for the flood itself; §5.7 shows
    how far it is from a 5 m solution.

### 5.7 Optional: 5 m references (`carlisle_peak_fine`)

A 5 m Cartesian grid of the same domain (582 012 cells) resolves the Eden's
banks. The case `carlisle_peak_fine` runs the same event on it with
local-inertial (GPU, Float64) and SWE (GPU, Float32). It needs a CUDA GPU and
the 5 m mesh of the benchmark suite, and takes about 1 h and 4 h, so it is
not run by this notebook; the cells below use it only if both runs are
complete, and nothing else in the notebook depends on them.

**How the 22 m runs are compared with them** (`FA.compare_to_fine`): each 5 m
cell is assigned to the 22 m cell (A5 or Cartesian) containing its centre,
and per 22 m cell the 5 m results give the stored volume, the share of the
cell wetter than 0.1 m (the area of its 5 m cells deeper than 0.1 m) and the
water level of that wet part (area-weighted). Against them, per 22 m model:
storage, the level difference on cells at least half wet in both (median
and rms), the wetted area at 0.1 m (SGS: from its tables; cell-mean: whole
cells), and the fuzzy CSI Σ A·min(f, f₅) / Σ A·max(f, f₅) of the wetted
fractions f. Only 22 m cells the 5 m grid covers (≥ 90% of their area) count.

In [ ]:
cfine = FC.case("carlisle_peak_fine")
FINE_DONE = all(FC.job_status(cfine, j) === :done for j in cfine.jobs)
if FINE_DONE
    F = Dict(j.name => FA.load_run(FC.job_output(cfine, j); label = j.name == "cart5_li" ? "5 m local-inertial" : "5 m SWE")
             for j in cfine.jobs)
    for n in ("cart5_li", "cart5_swe")
        s = F[n].info["summary"]
        @printf("%-20s %d cells, %d steps, %.0f min on the GPU; storage at 55.5 h %.2f Mm³\n", F[n].label, length(F[n].mesh),
                s["n_steps"], s["wall_simulation_s"] / 60, FA.stored_volume(F[n], FA.frame_at(F[n], T_PEAK; tol = 300)) / 1e6)
    end
else
    println("""The 5 m references have not been run (case carlisle_peak_fine: $(join(("$(j.name) $(FC.job_status(cfine, j))" for j in cfine.jobs), ", "))).
    To make them (a CUDA GPU; about 1 h for local-inertial and 4 h for SWE):
        julia --project=. benchmarks/run.jl                          # once: the benchmark suite builds the 5 m mesh
        julia --project=. cases/run.jl run carlisle_peak_fine
    then re-run this section.""")
end

In [ ]:
if FINE_DONE
    FINE = ("cart5_swe", "cart5_li")
    COARSE = [("A5 res 18", ["a5_swe", "a5_li", "a5_sgs"]), ("Cartesian 20 m", ["cart_swe", "cart_li"])]
    OWN = Dict(m => FA.assign_to_cells(F["cart5_li"].mesh, P[ns[1]].mesh) for (m, ns) in COARSE)
    FC5 = Dict((f, n, t) => FA.compare_to_fine(F[f], P[n], t; owner = OWN[m], tables = n == "a5_sgs" ? SGS_TABLES : nothing)
               for f in FINE for (m, ns) in COARSE for n in ns for t in T_WIN)
    short = Dict("a5_swe" => "SWE", "a5_li" => "local-inertial", "a5_sgs" => "SGS", "cart_swe" => "SWE", "cart_li" => "local-inertial")
    for (m, ns) in COARSE
        rows = Vector{String}[]
        for n in ns, t in T_WIN
            a, b = FC5[("cart5_swe", n, t)], FC5[("cart5_li", n, t)]
            push!(rows, [short[n], hrs(t), f2(a.storage_ratio), @sprintf("%+.2f (%.2f)", a.level_median, a.level_rms),
                         f2(a.wet_area / 1e6) * " (" * f2(a.wet_area_fine / 1e6) * ")", f2(a.csi),
                         f2(b.storage_ratio), @sprintf("%+.2f (%.2f)", b.level_median, b.level_rms),
                         f2(b.wet_area_fine / 1e6), f2(b.csi)])
        end
        display(htmltable(["22 m model", "time", "vs 5 m SWE: storage ratio", "level − 5 m, median (rms) m",
                           "wetted area km² (5 m)", "CSI", "vs 5 m LI: storage ratio", "level − 5 m, median (rms) m",
                           "5 m wetted area km²", "CSI"], rows; caption = "$m against the 5 m references"))
    end
    # the two 5 m runs against each other, on their own cells
    k1, k2 = FA.frame_at(F["cart5_li"], T_PEAK; tol = 300), FA.frame_at(F["cart5_swe"], T_PEAK; tol = 300)
    h1, h2 = FA.depth(F["cart5_li"], k1), FA.depth(F["cart5_swe"], k2)
    η1, η2 = FA.wse(F["cart5_li"], k1), FA.wse(F["cart5_swe"], k2)
    dl = [η1[i] - η2[i] for i in eachindex(h1) if h1[i] > 0.1 && h2[i] > 0.1]
    @printf("\n5 m local-inertial against 5 m SWE at 55.5 h: storage ratio %.2f; level where both are wetter than 0.1 m: median %+.2f m, rms %.2f m\n",
            FA.stored_volume(F["cart5_li"], k1) / FA.stored_volume(F["cart5_swe"], k2), median(dl), sqrt(mean(abs2, dl)))
end

**Water level along the river at the peak.** For each 250 m band of x
(east–west; the Eden flows from east to west, left to right here), the
water level of the wet cell with the lowest ground in that band — the river
channel's level, in every model on its own terms (5 m and cell-mean runs:
the cell's bed; SGS: the cell's lowest ground; wet: deeper than 0.1 m above
it). Bottom: the same, minus the 5 m SWE level. In the last band, at the
western edge of the domain, each model's level is set by its open boundary
(the 5 m runs draw down there), so differences there say nothing about the
river.

In [ ]:
"Per x band (m, local frame about (lon0, lat0)): the water level of the wet cell with the lowest ground."
function river_profile(r, t, lon0, lat0; band = 250.0, wet = 0.1)
    k = FA.frame_at(r, t; tol = 300)
    h = FA.depth(r, k); η = FA.wse(r, k)
    z = r.flow_model == "sgs" ? r.mesh.static["sgs_z_min"] : r.mesh.elevation
    x, _ = FA.local_xy(r.mesh, lon0, lat0)
    b = floor.(Int, x ./ band)
    best = Dict{Int,Int}()
    for i in eachindex(h)
        (isfinite(h[i]) && h[i] > wet && isfinite(z[i])) || continue
        j = get(best, b[i], 0)
        (j == 0 || z[i] < z[j]) && (best[b[i]] = i)
    end
    ks = sort(collect(keys(best)))
    return (x = (ks .+ 0.5) .* band, level = [η[best[q]] for q in ks], ground = [z[best[q]] for q in ks])
end
if FINE_DONE
    lon0, lat0 = mean(P["a5_swe"].mesh.lon), mean(P["a5_swe"].mesh.lat)
    prof = [("5 m SWE", F["cart5_swe"], (color = :black, linestyle = :solid, linewidth = 2.5)),
            ("5 m local-inertial", F["cart5_li"], (color = :black, linestyle = :dash, linewidth = 2)),
            ("22 m SWE (A5)", P["a5_swe"], (color = FP.SERIES[1], linestyle = :dash, linewidth = 2)),
            ("22 m local-inertial (A5)", P["a5_li"], (color = FP.SERIES[1], linestyle = :solid, linewidth = 2)),
            ("22 m SGS (A5)", P["a5_sgs"], (color = FP.SERIES[3], linestyle = :solid, linewidth = 2))]
    PR = Dict(l => river_profile(r, T_PEAK, lon0, lat0) for (l, r, _) in prof)
    ref = PR["5 m SWE"]; refat = Dict(zip(ref.x, ref.level))
    fig = Figure(size = (1000, 720))
    ax1 = Axis(fig[1, 1]; ylabel = "Water level (m)", xreversed = true,
               title = "Water level along the river at 55.5 h (lowest-ground wet cell per 250 m band)")
    ax2 = Axis(fig[2, 1]; xlabel = "x (m east of the domain centre; the Eden flows left to right)",
               ylabel = "Level − 5 m SWE (m)", xreversed = true)
    lines!(ax1, ref.x, ref.ground; color = FP.MUTED, linewidth = 1, label = "lowest wet ground, 5 m")
    hlines!(ax2, [0.0]; color = FP.MUTED, linewidth = 1)
    for (l, _, st) in prof
        p = PR[l]
        lines!(ax1, p.x, p.level; st..., label = l)
        sel = [i for i in eachindex(p.x) if haskey(refat, p.x[i])]
        lines!(ax2, p.x[sel], p.level[sel] .- [refat[p.x[i]] for i in sel]; st...)
        @printf("%-26s level − 5 m SWE along the river: median %+.2f m over %d bands\n", l,
                median(p.level[sel] .- [refat[p.x[i]] for i in sel]), length(sel))
    end
    linkxaxes!(ax1, ax2); hidexdecorations!(ax1; grid = false, ticks = false)
    rowsize!(fig.layout, 2, Relative(0.35))
    Legend(fig[3, 1], ax1; orientation = :horizontal, nbanks = 2, framevisible = false)
    display(fig)
end;

**What the 5 m references show** (if they have been run; plan §9.3a records
the same numbers):

1. **The 22 m SWE run is not a reference for the flood.** It stores about a
   quarter more than 5 m SWE at the peak (≈ 1.24 on A5, ≈ 1.12 on the
   Cartesian grid), with its water about 0.4 m higher on A5 (0.15 m on the
   Cartesian grid). The bar agreed
   for SGS compared it with that run.
2. **SGS is the 22 m model closest to the 5 m local-inertial solution**:
   about 0.83 of its storage, with its water a median of 0.1–0.15 m lower
   (rms about 0.2 m), against +0.25 m (rms about 0.35 m) and 1.12 of the
   storage for local-inertial at 22 m. That is what a sub-grid scheme for
   these equations should do.
3. **Local-inertial physics conveys more than SWE even at 5 m**: the 5 m
   local-inertial run stores about 0.84 of 5 m SWE, its water a few
   decimetres lower. That gap is the momentum physics the local-inertial
   equations leave out (advection, lateral shear at bank lines), not
   resolution — and it is why SGS, a faithful sub-grid form of
   local-inertial, ends up below 5 m SWE (about 0.70 of its storage, water
   about 0.5 m lower).
4. **22 m local-inertial is close to 5 m SWE through compensating errors**
   (≈ 0.94 of its storage, level ≈ −0.16 m on A5): its smoothed cell-mean
   bed under-conveys, which offsets the over-conveyance of its physics.
5. SGS wets less area than the 5 m runs (≈ 3.1 against 3.8–4.0 km² at the
   peak), consistent with its lower level.
6. **Along the river channel** the order is the same: from top to bottom 22 m
   SWE, 5 m SWE, 22 m local-inertial, 5 m local-inertial and SGS, with SGS
   close to 5 m local-inertial over most of the reach.

So against the best estimate of reality here (5 m SWE) SGS is further off
than 22 m local-inertial, for a reason that is not a sub-grid defect: it is
now a faithful sub-grid form of a model that conveys too much in deep, fast
flow. The SGS check of §5.5 is to be restated against the 5 m runs (SGS
against 5 m local-inertial for the sub-grid representation; every 22 m
model against 5 m SWE for reality).

## 6. The recorded test window: the first 6 h, with test rain (`carlisle`)

This is the comparison recorded in HYDRAULICS §15.10–15.11, which set the 0.3.0
defaults. It covers the first 6 h of the event, with the river starting dry, and
**adds 10 mm/h of uniform rain on every cell**.

**That rain is an artificial test forcing, not the 2005 event's rain.** It comes
from the damping series of HYDRAULICS §15.6–15.11, where it served to test thin
sheet flow over a real floodplain. Over 6 h it puts 60 mm, 0.88 Mm³, on the whole
domain. That is only 15% of the water added (the inflows bring 5.04 Mm³), but it
wets every cell.

### 6.1 How much of the 6 h flooding is the rain?

The `*_norain` runs repeat the local-inertial runs without the rain.

In [ ]:
rows = [[g, f2(m6["$(k)_li"]["wet_area_0p1_km2"]), f2(m6["$(k)_li_norain"]["wet_area_0p1_km2"]),
         f3(m6["$(k)_li"]["storage_m3"] / 1e6), f3(m6["$(k)_li_norain"]["storage_m3"] / 1e6)]
        for (g, k) in (("A5", "a5"), ("Cartesian", "cart"))]
htmltable(["local-inertial, 6 h", "area > 0.1 m with rain (km²)", "without rain (km²)",
           "storage with rain (Mm³)", "without rain (Mm³)"], rows; caption = "carlisle at 6 h: with and without the test rain")

In [ ]:
let fig = Figure(), runs = [S["a5_li"] S["a5_li_norain"]; S["cart_li"] S["cart_li_norain"]]
    titles = ["A5 local-inertial + 10 mm/h test rain" "A5 local-inertial, river only";
              "Cartesian local-inertial + 10 mm/h test rain" "Cartesian local-inertial, river only"]
    for i in 1:2, j in 1:2
        FP.fieldmap!(fig[i, j], runs[i, j], nothing; t = T6, dry_threshold = 0.1, colorrange = (0, 5), width = 430,
                     title = titles[i, j], subtitle = "t = 06:00 · depth > 0.1 m", colorbar = j == 2)
    end
    resize_to_layout!(fig)
    fig
end

At 6 h the river inflow alone floods little more than the channels and the
lowest ground beside them. With the test rain, the flooded area (deeper than
0.1 m) is about 2.4 times larger on both meshes: hollows all over the floodplain fill with
rain. The extensive 6 h "flooding" of this case is therefore mostly rain-on-grid,
not river flooding, and its maps should not be read as the event's flood extent.

### 6.2 Local-inertial against SWE at 6 h, with the test rain

The recorded result, from the metrics of `carlisle`:

In [ ]:
rows = [[S[n].label, f3(m6[n]["storage_m3"] / 1e6), haskey(m6[n], "storage_ratio") ? f3(m6[n]["storage_ratio"]) : "–",
         haskey(m6[n], "rms_dh") ? f3(m6[n]["rms_dh"]) : "–", haskey(m6[n], "csi") ? f2(m6[n]["csi"]) : "–",
         haskey(m6[n], "bias") ? f2(m6[n]["bias"]) : "–", @sprintf("%.0f", m6[n]["arrival_s"])] for n in ORDER]
display(htmltable(["run", "storage (Mm³)", "÷ SWE", "rms Δh (m)", "CSI", "bias", "arrival (s)"], rows;
                  caption = "carlisle at 6 h (inflows + 10 mm/h test rain). Recorded (HYDRAULICS §15.10–15.11): SWE 2.690 (A5) and 2.331 (Cartesian) Mm³; local-inertial 1.25× and 0.99× SWE. Arrival: 50 000 m³ out."))
s060 = FC.TOML.parsefile(joinpath(REPO, "cases", "expected", "v0.6.0", "carlisle_metrics.toml"))["a5_sgs"]
c = FA.compare_to_fine(S["a5_swe"], S["a5_sgs"], T6; owner = collect(1:length(S["a5_swe"].mesh)), tables = SGS_TABLES)
@printf("SGS at 6 h: %.3f Mm³ stored (0.6.0 scheme: %.3f), %.2f of SWE; %.3f Mm³ let out (SWE %.3f, local-inertial %.3f); arrival %.0f s (0.6.0: %.0f s)
",
        m6["a5_sgs"]["storage_m3"] / 1e6, s060["storage_m3"] / 1e6, c.storage_ratio, S["a5_sgs"].info["summary"]["vol_removed_m3"] / 1e6,
        S["a5_swe"].info["summary"]["vol_removed_m3"] / 1e6, S["a5_li"].info["summary"]["vol_removed_m3"] / 1e6,
        m6["a5_sgs"]["arrival_s"], s060["arrival_s"])
@printf("  like for like against A5 SWE: wetted area %.2f km² (SWE %.2f; the case's whole-cell count by SGS depth: %.2f), fuzzy CSI %.2f, level − SWE where both wet: median %+.2f m (%d cells)
",
        c.wet_area / 1e6, c.wet_area_fine / 1e6, m6["a5_sgs"]["wet_area_0p1_km2"], c.csi, c.level_median, c.n_level)
let fig = Figure(size = (820, 460))
    FP.storage_plot!(fig[1, 1], [S[n] for n in ("a5_li", "a5_swe", "a5_sgs", "cart_li", "cart_swe")];
                     title = "Water stored, carlisle (first 6 h, inflows + 10 mm/h test rain)")
    display(fig)
end;

In [ ]:
let fig = Figure()
    FP.difference_map!(fig[1, 1], S["a5_swe"], S["a5_li"]; t = T6, colorrange = (-1.5, 1.5), width = 460,
                       title = "A5: local-inertial − SWE, 6 h, + test rain")
    FP.difference_map!(fig[1, 2], S["cart_swe"], S["cart_li"]; t = T6, colorrange = (-1.5, 1.5), width = 460,
                       title = "Cartesian: local-inertial − SWE, 6 h, + test rain")
    resize_to_layout!(fig)
    display(fig)
end;
display(storage_by_class(S["a5_swe"], S["a5_li"], T6))
display(storage_by_class(S["cart_swe"], S["cart_li"], T6))

**What the 6 h result shows:**

- **Cartesian.** The local-inertial default stays within 1% of the reference.
- **A5.** It stores 25% more, because it lets about 0.68 Mm³ less water out of the
  domain. Its first 50 000 m³ of outflow arrives about 50 min later.
- **Where the A5 excess is.** It is spread over every depth class. About a third
  is a film of about 2 cm on the 24 000 cells that the reference has nearly dry,
  which is the test rain's sheet. The rest is on the shallow floodplain and in the
  channels, which are 0.4 m deeper on average.
- **The cause, per HYDRAULICS §15.10, is mostly the bed representation.** The
  local-inertial solver lets water cross a face only above the higher of the two
  cell beds, max(z_i, z_j), so the cells form a staircase. The second-order SWE
  reference rebuilds a sloping bed at each face. A thin film drains across a
  sloping face bed, but a step bed holds it in shallow hollows. A first-order SWE
  run, which also uses a step bed, shows the same pattern of excess, more
  strongly.

With the test rain, much of this window is exactly that kind of thin sheet flow,
so the 6 h ratio of 1.25 says as much about rain-on-grid films as about the
river. §5 tests the river itself, at its peak and without rain.

**SGS at 6 h.** It now stores about 1.76 Mm³, about two thirds of SWE and
half of local-inertial (0.6.0 scheme: 4.30 Mm³, more than either): it lets
water out of the domain much sooner and faster than the other runs (its
first 50 000 m³ about an hour before SWE's). Its wetted area from its tables
is much smaller than SWE's, and much smaller than the case's own count for
it (`wet_area_0p1_km2`, whole cells whose SGS depth exceeds 0.1 m: with the
rain on every cell, almost every cell holds 0.1 m in its lowest part). That
count is not comparable with the cell-mean runs' and is not used here.

## 7. Summary and pointers

**Summary.** Local-inertial against the full shallow-water reference, on the same
mesh:

| | Cartesian 20 m | A5 res 18 |
|---|---|---|
| **Near the peak (55.5 h, river only)**: storage ÷ SWE | 0.85 | 0.76 |
| rms Δh | 0.34 m | 0.54 m |
| CSI, bias | 0.92, 0.93 | 0.87, 0.89 |
| **First 6 h, with the 10 mm/h test rain**: storage ÷ SWE | 0.99 | 1.25 |
| rms Δh | 0.13 m | 0.29 m |
| CSI, bias | 0.69, 0.85 | 0.62, 1.06 |

- **The test rain dominates the first 6 h.** It is an artificial test forcing,
  not the event's rain, and it makes most of that window's flooded area: about
  2.4 times the area the river floods on its own.
- **Near the peak the local-inertial default stores less than the reference.**
  Its deep water is faster and it floods slightly less: 15% less storage on
  Cartesian and 24% less on A5. Early in the event, with the rain, its A5 run
  stored 25% *more*. The sign of its error depends on the flow.
- **The mesh.** Near the peak the local-inertial runs hardly differ between the
  meshes. The two references differ by 12%, the A5 one holding the Eden higher.
- **SGS** (rewritten in 0.7.0). Near the peak it stores about 0.55 of the
  22 m SWE run and its water stands about 0.9 m lower where both are wet
  (until 0.6.0: 2.5 times SWE, metres higher); it is smooth and converged in
  its time step at its default Courant number 0.5. It conveys more than
  local-inertial, so the check of the bar agreed for it fails and is an
  expected failure. Against 5 m references (§5.7, optional) it is the 22 m
  model closest to 5 m local-inertial, and the 22 m SWE run is itself about
  a quarter above 5 m SWE: the bar is to be restated against the 5 m runs.
- **Mass.** It is conserved to round-off in every run.

**Replaying a run** in the interactive GLMakie viewer, with a frame slider and
the live viewer's controls. It needs the full stack (`--project=.`, without
`FLOODA5_NO_VIS`):

```bash
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle_peak/runs/a5_li.h5 --from 178200
julia --project=. --threads auto visualisation/replay.jl test/case_runs/carlisle_peak/runs/a5_swe.h5 --var velocity --from 199800
```

**From the shell:**

```bash
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl show carlisle_peak            # description and commands
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl run carlisle carlisle_peak     # prepare, run, check
FLOODA5_NO_VIS=1 julia --project=. cases/run.jl check carlisle_peak            # check existing runs
julia --project=. FloodModel.jl --rerun test/case_runs/carlisle_peak/runs/a5_li.h5   # reproduce one run
```

**Further reading:**

- `docs/TEST_CASES.md` §3: the Carlisle cases.
- `docs/HYDRAULICS.md` §9: the SGS solver (0.7.0); plan
  `projectinstructions/FloodA5_SGS_Conveyance_Fix_Plan_2026-11.md` §9.3–9.3a:
  SGS at the peak and the 5 m comparison.
- `docs/HYDRAULICS.md`:
  - §15.10: the SWE reference and the bed-representation analysis;
  - §15.11: the 0.3.0 defaults;
  - §15.12: the channel series.
- `docs/DATA_FORMATS.md`: the output file (§2) and the `.bci`/`.bdy` formats (§3).
- `docs/USER_GUIDE.md` §6.5: the figure functions used here.